# FaceVox training

Train with the same dataset preparation, models and checkpoint format as the CLI.
In Colab, select a GPU runtime if using a PyTorch model. The setup cell clones the
repository only when it is not already available.


In [ ]:
from pathlib import Path
import json
import os
import subprocess
import sys

if not Path("src/utils/training.py").is_file():
    repo = Path.cwd() / "facevox"
    if not (repo / ".git").is_dir():
        subprocess.run(["git", "clone", "https://github.com/valleysonata/facevox.git", str(repo)], check=True)
    os.chdir(repo)

subprocess.run([sys.executable, "-m", "pip", "install", "-r", "requirements.txt"], check=True)
subprocess.run(["git", "rev-parse", "HEAD"], check=True)


## Data and configuration

Put captured `dataset*.json` files in the repository's `data/` directory, using
Colab's file browser or a local checkout. Files uploaded to Colab are stored in
that Google runtime; use local training if the recordings must stay on your machine.

Leave `SYNTHETIC = False` for captured data. Synthetic mode only checks that the
training pipeline runs; its scores are not real-face benchmarks.


In [ ]:
MODEL_TYPE = "transformer"  # rf, gb, transformer, temporal, occlusion_aware
EPOCHS = 30
SYNTHETIC = False
SAMPLES_PER_CLASS = 50


In [ ]:
import numpy as np
import torch
from src.utils.training import ExpressionDatasetBuilder, ExpressionTrainer

np.random.seed(42)
torch.manual_seed(42)
print("Device:", "CUDA" if torch.cuda.is_available() else "CPU")

dataset = ExpressionDatasetBuilder(output_dir="data")
if SYNTHETIC:
    dataset.generate_synthetic(samples_per_class=SAMPLES_PER_CLASS)
else:
    dataset.merge_all()
if not dataset.data:
    raise ValueError("No dataset*.json files found in data/. Upload captured data or explicitly select SYNTHETIC mode.")

trainer = ExpressionTrainer(model_type=MODEL_TYPE)
X, y = trainer.prepare_data(dataset)
print(f"Samples: {len(y)}, features: {X.shape[1]}")


## Training and metrics

The shared trainer uses a random sample split. For PyTorch, this same split is
used for checkpoint selection, so the reported score is a validation score rather
than an independent test benchmark. Sessions and subjects are not held apart.


In [ ]:
metrics = trainer.train(X, y, epochs=EPOCHS)
print(json.dumps(metrics, indent=2))


## Save the checkpoint

Download the model and metrics from `checkpoints/` in Colab's file browser. The
model filename matches what the CLI expects; copy it into the local repository's
`checkpoints/` directory to use it with the webcam demo.


In [ ]:
torch_model = MODEL_TYPE in {"transformer", "temporal", "occlusion_aware"}
model_path = Path("checkpoints") / (f"expression_{MODEL_TYPE}.pt" if torch_model else "expression_model.joblib")
model_path.parent.mkdir(exist_ok=True)
trainer.save(str(model_path))
metrics_path = model_path.with_suffix(".metrics.json")
metrics_path.write_text(json.dumps({"synthetic": SYNTHETIC, "metrics": metrics}, indent=2), encoding="utf-8")
print("Saved:", model_path, metrics_path)
